# IBM AML Demo Queries

This notebook runs Gremlin queries and shows query results.
Start backend first in another terminal:
`mvn exec:java`


In [ ]:
import requests
import pandas as pd
import subprocess
import json as _json
from typing import Dict, Any
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
MAX_ROWS = 100000
REPO_ROOT = Path.home() / "SourceCode/graph-query-engine"
SHOW_PLAN = False


def run_aml_data_download(variant: str = "HI-Small", rows: int = 100000) -> bool:
    # Download HI-Small files and generate demo/data/aml-demo.csv via normalize_aml.py.
    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    cmd = f"bash ./scripts/download_aml_data.sh --variant {variant} --rows {rows}"
    display(Markdown(f"Running: `{cmd}` in `{REPO_ROOT}`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.returncode == 0


def resolve_csv_path() -> str | None:
    candidates = [
        Path.cwd() / "demo/data/aml-demo.csv",
        Path.cwd() / "demo/data/HI-Small_Trans.csv",
        Path.cwd() / "data/aml-normalized.csv",
        Path.cwd() / "data/aml-demo.csv",
        Path.cwd() / "aml-demo.csv",
    ]
    for p in candidates:
        if p.exists():
            return str(p)

    # Fallback to any normalized or raw transaction CSV in demo/data.
    demo_data = Path.cwd() / "demo/data"
    if demo_data.exists():
        for pat in ["*aml*.csv", "*Trans.csv"]:
            matches = sorted(demo_data.glob(pat))
            if matches:
                return str(matches[0])
    return None


CSV_PATH = resolve_csv_path()
print("BASE_URL:", BASE_URL)
print("CSV_PATH:", CSV_PATH)
print("MAX_ROWS:", MAX_ROWS)


## Step 1: Prepare CSV (HI-Small)

If `demo/data/aml-demo.csv` is missing, run this in terminal:

```zsh
cd ~/SourceCode/graph-query-engine
bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000
```

This downloads `HI-Small_Trans.csv` and creates normalized `demo/data/aml-demo.csv`.

You can also run the helper below from notebook.


In [ ]:
AUTO_RUN_DOWNLOAD = False
DOWNLOAD_VARIANT = "HI-Small"
DOWNLOAD_ROWS = 100000

if AUTO_RUN_DOWNLOAD:
    ok = run_aml_data_download(variant=DOWNLOAD_VARIANT, rows=DOWNLOAD_ROWS)
    if ok:
        CSV_PATH = resolve_csv_path()
        print("CSV_PATH refreshed:", CSV_PATH)
else:
    print("Set AUTO_RUN_DOWNLOAD=True to run download + normalize from notebook.")


In [ ]:
def get_sql_explain(gremlin_query: str, include_plan: bool = False) -> Dict[str, Any]:
    try:
        response = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            params={"plan": "true"} if include_plan else {},
            timeout=10,
        )
        if response.ok:
            return response.json()
        return {"error": f"HTTP {response.status_code}: {response.text}"}
    except Exception as e:
        return {"error": str(e)}


def run_gremlin_query(gremlin_query: str, tx_mode: bool = False) -> Dict[str, Any]:
    endpoint = "/gremlin/query/tx" if tx_mode else "/gremlin/query"
    result: Dict[str, Any] = {}
    try:
        response = requests.post(
            f"{BASE_URL}{endpoint}",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            timeout=60,
        )
        result = response.json()
    except Exception as e:
        result = {"error": str(e)}
    result["_sql_explain"] = get_sql_explain(gremlin_query, include_plan=SHOW_PLAN)
    return result


def display_query_result(gremlin: str, result: Dict[str, Any], title: str = "", limit: int = 10, tx_mode: bool = False):
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown("**Gremlin:**"))
    display(Markdown(f"```groovy\n{gremlin}\n```"))

    explain = result.get("_sql_explain", {})
    if "error" in explain:
        display(Markdown(f"**SQL Translation:** *not available ({explain['error']})*"))
    else:
        sql = explain.get("translatedSql", "")
        params = explain.get("parameters", [])
        plan = explain.get("plan")
        if sql:
            display(Markdown("**SQL Translation:**"))
            display(Markdown(f"```sql\n{sql}\n```"))
            if params:
                display(Markdown(f"**Parameters:** `{params}`"))
        if plan:
            display(Markdown("**Query Plan:**"))
            display(Markdown(f"```json\n{_json.dumps(plan, indent=2)}\n```"))

    if "error" in result:
        display(Markdown(f"**Error:** {result['error']}"))
        return

    rows = result.get("results", [])
    display(Markdown(f"**Result Count:** {result.get('resultCount', 0)}"))
    if not rows:
        return
    if isinstance(rows[0], dict):
        display(pd.DataFrame(rows[:limit]))
    else:
        for i, row in enumerate(rows[:limit], 1):
            print(f"{i}. {row}")


## Step 0: Health check


In [ ]:
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).text
    provider = requests.get(f"{BASE_URL}/gremlin/provider", timeout=5).json().get("provider", "unknown")
    display(Markdown(f"Status: `{health}`"))
    display(Markdown(f"Provider: `{provider}`"))
except Exception as e:
    display(Markdown(f"Health check failed: {e}"))


## Step 2: Load AML CSV and Mapping


In [ ]:
if not CSV_PATH:
    display(Markdown("**CSV not found.**"))
    display(Markdown('''Expected one of:
- `demo/data/aml-demo.csv`
- `demo/data/HI-Small_Trans.csv`
- `data/aml-normalized.csv`
- `data/aml-demo.csv`'''))
    display(Markdown('''Use:
```zsh
cd ~/SourceCode/graph-query-engine
bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000
```'''))
else:
    response = requests.post(
        f"{BASE_URL}/admin/load-aml-csv",
        params={"path": CSV_PATH, "maxRows": MAX_ROWS},
        timeout=120,
    )
    print(response.json())


## Query Sections


## Simple Queries


### S1 Count accounts

Count unique Account vertices.


In [ ]:
gremlin = "g.V().hasLabel('Account').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S1 Count accounts', limit=10, tx_mode=False)


### S2 Count banks

Count Bank vertices - one per distinct bank ID in the data.


In [ ]:
gremlin = "g.V().hasLabel('Bank').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S2 Count banks', limit=10, tx_mode=False)


### S3 Count countries

Count Country vertices (10 pre-seeded jurisdictions).


In [ ]:
gremlin = "g.V().hasLabel('Country').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S3 Count countries', limit=10, tx_mode=False)


### S4 Count alerts

Count Alert vertices - one raised per suspicious transfer.


In [ ]:
gremlin = "g.V().hasLabel('Alert').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S4 Count alerts', limit=10, tx_mode=False)


### S5 Count transfers

Count all TRANSFER edges.


In [ ]:
gremlin = "g.E().hasLabel('TRANSFER').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S5 Count transfers', limit=10, tx_mode=False)


### S6 Suspicious transfer count

Count confirmed suspicious TRANSFER edges (isLaundering=1).


In [ ]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S6 Suspicious transfer count', limit=10, tx_mode=False)


### S7 High-risk countries

List Country vertices with riskLevel=HIGH.


In [ ]:
gremlin = "g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S7 High-risk countries', limit=10, tx_mode=False)


### S8 High-severity alerts

Show HIGH-severity open alerts.


In [ ]:
gremlin = "g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S8 High-severity alerts', limit=15, tx_mode=False)


## Complex Queries


### C1 Top sender accounts

Accounts ranked by outgoing transfer count - find the biggest hubs.


In [ ]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C1 Top sender accounts', limit=15, tx_mode=False)


### C2 Suspicious hubs

Accounts with the most suspicious outgoing transfers.


In [ ]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C2 Suspicious hubs', limit=15, tx_mode=False)


### C3 Account -> Bank (BELONGS_TO)

Show which bank each account belongs to via BELONGS_TO.


In [ ]:
gremlin = "g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C3 Account -> Bank (BELONGS_TO)', limit=15, tx_mode=False)


### C4 Bank -> Country (LOCATED_IN)

Show which country each bank is headquartered in.


In [ ]:
gremlin = "g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C4 Bank -> Country (LOCATED_IN)', limit=15, tx_mode=False)


### C5 Two-hop: Account -> Bank -> Country

Traverse Account->Bank->Country in two hops.


In [ ]:
gremlin = "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C5 Two-hop: Account -> Bank -> Country', limit=10, tx_mode=False)


### C6 Accounts sending to high-risk countries (SENT_VIA)

Find accounts routing money via FATF-blacklisted countries.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C6 Accounts sending to high-risk countries (SENT_VIA)', limit=20, tx_mode=False)


### C7 Flagged accounts with alert detail (FLAGGED_BY)

Show investigated accounts with linked Alert severity.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C7 Flagged accounts with alert detail (FLAGGED_BY)', limit=20, tx_mode=False)


### C8 Cross-bank suspicious flow

Suspicious transfers that cross bank boundaries.


In [ ]:
gremlin = "g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C8 Cross-bank suspicious flow', limit=15, tx_mode=False)


### C9 Three-hop money trail

Follow suspicious TRANSFER hops 3 steps deep.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C9 Three-hop money trail', limit=10, tx_mode=False)


### C10 Five-hop layering chain

Extended 5-hop traversal for layering detection.


In [ ]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C10 Five-hop layering chain', limit=10, tx_mode=False)


### C11 Transactional suspicious count

Suspicious transfer count via transactional endpoint.


In [ ]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=True)
display_query_result(gremlin, result, title='C11 Transactional suspicious count', limit=10, tx_mode=True)


## Iceberg Note

Use `aml_iceberg_queries.ipynb` for Iceberg comparisons.


## Done
